# Kilter climb generator — training (Colab / Kaggle GPU)

Trains the conditional transformer (`scripts/train_generator.py`) from the Snappet repo on a GPU.
It builds the size-aware, match/no-match-conditioned dataset from the bundled Kilter snapshot, trains a small
GPT over the `[BOS][SIZE][ANGLE][GRADE][MATCH] … [EOS]` sequences, then samples climbs under the
per-size hold mask. The trained checkpoint is what gets exported to ONNX and wired into the web app's
BoardView. **Runtime → Change runtime type → GPU** before running.

Background: `pdd/context/research/kilter-climb-generator.md`.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — set Runtime to GPU')

In [ ]:
# Clone the repo (includes the bundled Kilter snapshot + the scripts).
# NOTE: these scripts currently live on the feature branch; switch BRANCH to 'main' once merged.
BRANCH = 'claude/upbeat-hypatia-QmdRe'
%cd /content
!rm -rf Snappet
!git clone --depth 1 --branch $BRANCH https://github.com/harshal2802/Snappet.git
%cd Snappet

In [ ]:
# Build the dataset from the bundled snapshot (~66k crowd-validated examples, a few seconds).
!python3 scripts/build-climb-dataset.py --min-ascents 5

In [ ]:
# Train the generator. ~6M params; a handful of minutes on a T4. Tune freely.
!python3 scripts/train_generator.py --epochs 12 --dim 256 --layers 6 --heads 8 --batch 256

In [ ]:
# Optional: train the grade predictor so generation can be re-ranked onto the target grade.
!python3 scripts/train_grade_predictor.py

In [ ]:
# Sample a climb: 12x12 (size 10), 40 degrees, grade ~20 (V5), match.
!python3 scripts/train_generator.py --sample --size 10 --angle 40 --grade 20 --nomatch 0
# A harder no-match one at 45 degrees:
!python3 scripts/train_generator.py --sample --size 10 --angle 45 --grade 23 --nomatch 1

## Next steps

- **Evaluate** with the research-doc metrics: validity (≈100% by construction), fits-requested-size, grade MAE
  (run `train_generator.py --sample` N times and score with `train_grade_predictor.py --score`), novelty
  (Jaccard to training), diversity — stratified by size and match/no-match.
- **Export to ONNX** for in-browser inference (`onnxruntime-web` / `transformers.js`), then wire into the
  Board Explorer's `BoardView` as a "Generate" tab.
- Download the checkpoint: `scripts/out/climb-dataset/generator.pt`.